# TN1 phần A — LSTM làm mốc

Chạy **song song** với `TN1_TCN_DSTCN_model_selection.ipynb` ở một phiên Colab khác. Hai notebook độc lập hoàn toàn, không cần chờ nhau.

| notebook | chạy gì | thời gian |
|---|---|---|
| **TN1_LSTM.ipynb** ← đang mở | LSTM: 4 fold CV, rồi 3 seed test GHIJ | ~3 giờ |
| TN1_TCN_DSTCN_model_selection.ipynb | TCN-64 và DS-TCN-64: 4 fold CV mỗi cái | ~2.2 giờ |
| TN1_final_evaluation.ipynb | gộp kết quả, so sánh, chạy GHIJ cho kiến trúc thắng | ~1.5 giờ |

## Vì sao LSTM phải chạy CV

Số LSTM ở TN0 (`0.822642`) đo trên **G H I J**, train đủ **8 người**. Số TN1 đo trên **4 fold của A B C D E F K L**, mỗi model train **6 người**. Khác người test, khác lượng dữ liệu, khác giao thức — đặt cạnh nhau là vô nghĩa.

Muốn nói "TCN hơn LSTM" thì LSTM phải được đo bằng đúng cái thước đó.

## Giao thức

Bốn fold cố định trên tám người `A B C D E F K L`, dùng y nguyên cho mọi thí nghiệm:

```
val_AB   train C D E F K L    chấm A B
val_CE   train A B D F K L    chấm C E
val_DF   train A B C E K L    chấm D F
val_KL   train A B C D E F    chấm K L
```

Cấu hình giữ nguyên như MobiVital công bố: 20 epoch, Adam lr 1e-4, batch 64, MSE, `corr_threshold` 0.9, không RevIN. Một seed.

Điểm chấm trên **buổi ghi thô**, model tự chọn kênh, không nhìn nhịp thở thật.

## 1. Chuẩn bị Colab

Mount Drive để lấy lại cửa sổ train đã cắt ở `DATA_PREPARE.ipynb`.

In [3]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


Tải mã nguồn rồi vào thư mục đó. `setup_colab.py` clone MobiVital và ghim commit `4319731d` — `src/mobivital_reference.py` mượn sáu hàm từ repo tác giả.

In [1]:
# Phải clone repo trước, vì setup_colab.py nằm bên trong chính repo đó.
# Xoá trước để chạy lại ô này luôn lấy mã mới nhất, không dính bản cũ.
!rm -rf /content/UWB_RADAR
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py

/content/UWB_RADAR

thư mục làm việc : /content/UWB_RADAR
commit đồ án     : 490027e
commit MobiVital : 4319731 (đã ghim)
GPU              : NVIDIA L4, 23034 MiB


Lấy `by_user/` và `windows/` từ Drive. Không cần CSV thô 13 GB — chỉ train trên cửa sổ đã cắt, chấm trên `by_user/*.npz`.

In [4]:
!python scripts/restore_processed_data_on_drive.py

by_user   : bung /content/drive/MyDrive/mobivital/by_user.tar ...
            12 tệp
windows   : bung /content/drive/MyDrive/mobivital/windows.tar.gz ...
            dev_cv 8 tệp, final_train có

2.5G	data/processed/by_user
503M	data/processed/windows


## 2. LSTM — 4 fold CV

Ghi 5 dòng vào `runs/summary.csv`: bốn dòng fold và một dòng `TONG` mang `cv_score` cùng `cv_std`.

Khoảng **1.5 giờ**: 0.85 phút mỗi epoch × 20 epoch × 4 fold, cộng chấm điểm 1289 buổi ghi.

In [5]:
!python scripts/run_cv.py --experiment tn1 --model lstm

thực nghiệm tn1  -> runs/tn1/
cấu hình lstm_mse_corr0.9_seed0
thiết bị  NVIDIA L4

----------------------------------------------------------
val_AB  train CDEFKL  chấm AB
210964 cửa sổ train
epoch  0  mse 0.03812  pearson 0.4898   0.8 phút
epoch  1  mse 0.02258  pearson 0.5665   1.6 phút
epoch  2  mse 0.02034  pearson 0.5858   2.4 phút
epoch  3  mse 0.01908  pearson 0.5987   3.2 phút
epoch  4  mse 0.01839  pearson 0.6069   4.0 phút
epoch  5  mse 0.01771  pearson 0.6147   4.8 phút
epoch  6  mse 0.01718  pearson 0.6194   5.6 phút
epoch  7  mse 0.01677  pearson 0.6229   6.4 phút
epoch  8  mse 0.01636  pearson 0.6273   7.2 phút
epoch  9  mse 0.01604  pearson 0.6316   8.0 phút
epoch 10  mse 0.01575  pearson 0.6354   8.8 phút
epoch 11  mse 0.01542  pearson 0.6384   9.6 phút
epoch 12  mse 0.01511  pearson 0.6408   10.4 phút
epoch 13  mse 0.01487  pearson 0.6440   11.2 phút
epoch 14  mse 0.01456  pearson 0.6465   12.0 phút
epoch 15  mse 0.01434  pearson 0.6485   12.8 phút
epoch 16  mse 0.0140

## 3. LSTM — mốc GHIJ, 3 seed

Train đủ tám người `A B C D E F K L` rồi test 537 buổi ghi của `G H I J` — đúng pipeline bài báo dùng, không phải model fold chỉ train 6 người.

Ba seed để báo cáo `mean ± std`. Một lần chạy cho một con số không phân biệt được hơn thật với hơn may: trọng số khởi tạo và thứ tự xáo trộn đổi theo seed.

Bước này **không phụ thuộc kết quả CV** nên chạy luôn được. Kiến trúc thắng sẽ chạy GHIJ ở `TN1_final_evaluation.ipynb` sau.

Khoảng **1.5 giờ**.

In [6]:
!python scripts/run_final_test.py --experiment tn1_ghij --model lstm --seed 0
!python scripts/run_final_test.py --experiment tn1_ghij --model lstm --seed 1
!python scripts/run_final_test.py --experiment tn1_ghij --model lstm --seed 2

thực nghiệm tn1_ghij  -> runs/tn1_ghij/
run_id   lstm_mse_corr0.9_seed0
thiết bị NVIDIA L4

292708 cửa sổ train
1502713 tham số

epoch  0  mse 0.03710  pearson 0.5285   1.1 phút
epoch  1  mse 0.02307  pearson 0.5887   2.2 phút
epoch  2  mse 0.02100  pearson 0.6065   3.3 phút
epoch  3  mse 0.01983  pearson 0.6173   4.5 phút
epoch  4  mse 0.01904  pearson 0.6280   5.6 phút
epoch  5  mse 0.01837  pearson 0.6341   6.7 phút
epoch  6  mse 0.01784  pearson 0.6400   7.8 phút
epoch  7  mse 0.01735  pearson 0.6439   8.9 phút
epoch  8  mse 0.01688  pearson 0.6496   10.0 phút
epoch  9  mse 0.01647  pearson 0.6535   11.1 phút
epoch 10  mse 0.01608  pearson 0.6580   12.2 phút
epoch 11  mse 0.01578  pearson 0.6609   13.4 phút
epoch 12  mse 0.01546  pearson 0.6632   14.5 phút
epoch 13  mse 0.01522  pearson 0.6661   15.6 phút
epoch 14  mse 0.01495  pearson 0.6683   16.7 phút
epoch 15  mse 0.01472  pearson 0.6711   17.8 phút
epoch 16  mse 0.01448  pearson 0.6737   18.9 phút
epoch 17  mse 0.01427  pearso

## 4. Cất kết quả

Nén ra tên riêng `tn1_lstm.zip` để **không đè** tệp của phiên chạy TCN. Cả hai phiên đều ghi `runs/tn1/`; tệp điểm không đè nhau vì tên cấu hình khác nhau, nhưng `summary.csv` thì mỗi phiên chỉ có dòng của riêng nó.

In [7]:
!cd runs && zip -qr /content/drive/MyDrive/mobivital/tn1_lstm.zip tn1 tn1_ghij summary.csv
!ls -la /content/drive/MyDrive/mobivital/

total 2726386
-rw------- 1 root root 2640629760 Sep  3 15:32 by_user.tar
-rw------- 1 root root    5643034 Sep  4 00:57 tn0.zip
-rw------- 1 root root   39092589 Sep  4 16:57 tn1_lstm.zip
-rw------- 1 root root  106452993 Sep  3 15:29 windows.tar.gz
